# Classification
This notebook contains the code for initial classification experiments, as well as out-of-domain tests for the MSc on Human vs AI Jazz.

### In-domain classification

In [ ]:
import os
import warnings
import joblib

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.model_selection import StratifiedGroupKFold

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier

import shap

In [ ]:
warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

MASTER_DATASET = # final master dataset containing all feature sets and sv groups
OUTPUT_DIR =

RANDOM_STATE = 42

OUTER_FOLDS = 5

PERMUTATION_REPEATS = 10

SHAP_SAMPLE_SIZE = 500

# ============================================================
# OUTPUT DIRS
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

for subdir in [
    "metrics",
    "models",
    "predictions",
    "permutation",
    "shap"
]:
    os.makedirs(
        os.path.join(OUTPUT_DIR, subdir),
        exist_ok=True
    )

# ============================================================
# LOAD DATA
# ============================================================

master_df = pd.read_parquet(MASTER_DATASET)

print("Loaded:", master_df.shape)

# ============================================================
# LABELS
# ============================================================

master_df["label"] = (
    master_df["flag"] == "ai"
).astype(int)

# ============================================================
# FEATURE SETS
# ============================================================

librosa_cols = [
    c for c in master_df.columns
    if c.startswith("librosa_")
]

rqa_cols = [
    c for c in master_df.columns
    if c.startswith("rqa_")
]
# ============================================================
# CLEAN RQA FEATURES
# ============================================================

master_df[rqa_cols] = master_df[rqa_cols].replace(
    [np.inf, -np.inf],
    np.nan
)

master_df[rqa_cols] = master_df[rqa_cols].fillna(
    master_df[rqa_cols].median()
)

prefixes = ('rqa_RR_')
suffixes = ('_median')

cols_to_drop = [col for col in master_df.columns if col.startswith(prefixes)]
cols_to_drop += [col for col in master_df.columns if col.endswith(suffixes)]

master_df.drop(columns=cols_to_drop, inplace=True)

rqa_cols = [c for c in master_df.columns if c.startswith("rqa_")]

clap_cols = [
    c for c in master_df.columns
    if c.startswith("clap_")
]

FEATURE_SETS = {

    "librosa":
        librosa_cols,

    "rqa":
        rqa_cols,

    "clap":
        clap_cols,

    'all':
        librosa_cols + rqa_cols + clap_cols,

    'librosa_rqa':
        librosa_cols + rqa_cols
}

# ============================================================
# MODELS
# ============================================================

MODELS = {

    "logreg":

        Pipeline([

            ("scaler", StandardScaler()),

            ("clf",
                LogisticRegression(
                    C=1.0,
                    max_iter=3000,
                    penalty="l2",
                    solver="lbfgs",
                    random_state=RANDOM_STATE
                )
            )
        ]),

    "svm":

        Pipeline([

            ("scaler", StandardScaler()),

            ("clf",
                SVC(
                    kernel="linear",
                    C=1.0,
                    probability=True,
                    random_state=RANDOM_STATE
                )
            )
        ]),

    "xgboost":

        Pipeline([

            ("clf",
                XGBClassifier(
                    max_depth=5,
                    learning_rate=0.05,
                    n_estimators=200,
                    eval_metric="logloss",
                    tree_method="hist",
                    device="cuda",
                    random_state=RANDOM_STATE
                )
            )
        ])
}

# ============================================================
# STORAGE
# ============================================================

all_metrics = []

# ============================================================
# MAIN LOOP
# ============================================================

for feature_set_name, feature_cols in FEATURE_SETS.items():

    print("\n" + "=" * 80)
    print("FEATURE SET:", feature_set_name)
    print("=" * 80)

    X = master_df[feature_cols]

    y = master_df["label"]

    # ========================================================
    # USE PRECOMPUTED OUTER FOLDS
    # ========================================================

    OUTER_FOLD_COL = "outer_fold"

    fold_ids = sorted(
        master_df[OUTER_FOLD_COL].dropna().unique()
    )

    print("Using saved folds:", fold_ids)

    # ========================================================
    # MODELS
    # ========================================================

    for model_name, model in MODELS.items():

        print("\n" + "-" * 80)
        print("MODEL:", model_name)
        print("-" * 80)

        # ====================================================
        # FOLDS
        # ====================================================

        for fold in fold_ids:

            print(f"\nFOLD: {fold}")

            # ================================================
            # TRAIN/TEST INDICES FROM SAVED FOLDS
            # ================================================

            test_mask = (
                master_df[OUTER_FOLD_COL] == fold
            )

            test_idx = np.where(test_mask)[0]

            train_idx = np.where(~test_mask)[0]

            print(f"\nFOLD: {fold}")

            # =================================================
            # SPLIT
            # =================================================

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]

            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]

            # =================================================
            # TRAIN
            # =================================================

            model.fit(X_train, y_train)

            # =================================================
            # PREDICT
            # =================================================

            y_pred = model.predict(X_test)

            y_prob = model.predict_proba(X_test)[:, 1]

            # =================================================
            # METRICS
            # =================================================

            metrics = {

                "feature_set":
                    feature_set_name,

                "model":
                    model_name,

                "fold":
                    fold,

                "roc_auc":
                    roc_auc_score(y_test, y_prob),

                "accuracy":
                    accuracy_score(y_test, y_pred),

                "balanced_accuracy":
                    balanced_accuracy_score(
                        y_test,
                        y_pred
                    ),

                "precision":
                    precision_score(y_test, y_pred),

                "recall":
                    recall_score(y_test, y_pred),

                "f1":
                    f1_score(y_test, y_pred)
            }

            all_metrics.append(metrics)

            print(metrics)

            # =================================================
            # SAVE MODEL
            # =================================================

            model_path = os.path.join(

                OUTPUT_DIR,

                "models",

                f"{feature_set_name}_{model_name}_fold{fold}.joblib"
            )

            joblib.dump(model, model_path)

            # =================================================
            # SAVE PREDICTIONS
            # =================================================

            pred_df = pd.DataFrame({

                "filename":
                    master_df.iloc[test_idx].index,

                "y_true":
                    y_test.values,

                "y_pred":
                    y_pred,

                "y_prob":
                    y_prob
            })

            pred_path = os.path.join(

                OUTPUT_DIR,

                "predictions",

                f"{feature_set_name}_{model_name}_fold{fold}.csv"
            )

            pred_df.to_csv(pred_path, index=False)

            # =================================================
            # PERMUTATION IMPORTANCE
            # =================================================

            print("Computing permutation importance...")

            perm = permutation_importance(

                model,

                X_test,

                y_test,

                n_repeats=PERMUTATION_REPEATS,

                random_state=RANDOM_STATE,

                scoring="roc_auc",

                n_jobs=-1
            )

            perm_df = pd.DataFrame({

                "feature":
                    X_test.columns,

                "importance_mean":
                    perm.importances_mean,

                "importance_std":
                    perm.importances_std
            })

            perm_path = os.path.join(

                OUTPUT_DIR,

                "permutation",

                f"{feature_set_name}_{model_name}_fold{fold}.csv"
            )

            perm_df.to_csv(perm_path, index=False)

            # =================================================
            # SHAP
            # =================================================

            print("Computing SHAP...")

            if len(X_test) > SHAP_SAMPLE_SIZE:

                shap_idx = np.random.choice(

                    len(X_test),

                    SHAP_SAMPLE_SIZE,

                    replace=False
                )

                X_shap = X_test.iloc[shap_idx]

            else:

                X_shap = X_test

            # =================================================
            # XGBOOST SHAP
            # =================================================

            if model_name == "xgboost":

                clf = model.named_steps["clf"]

                explainer = shap.TreeExplainer(clf)

                shap_values = explainer.shap_values(X_shap)

            # =================================================
            # LINEAR MODEL SHAP
            # =================================================

            else:

                scaler = model.named_steps["scaler"]

                clf = model.named_steps["clf"]

                X_train_scaled = scaler.transform(X_train)

                X_shap_scaled = scaler.transform(X_shap)

                explainer = shap.LinearExplainer(

                    clf,

                    X_train_scaled
                )

                shap_values = explainer.shap_values(
                    X_shap_scaled
                )

            # =================================================
            # SAVE SHAP
            # =================================================

            shap_output = {

                "shap_values":
                    shap_values,

                "feature_names":
                    X_shap.columns.to_list(),

                "filenames":
                    X_shap.index.to_numpy()
            }

            shap_path = os.path.join(

                OUTPUT_DIR,

                "shap",

                f"{feature_set_name}_{model_name}_fold{fold}.joblib"
            )

            joblib.dump(shap_output, shap_path)

# ============================================================
# SAVE ALL METRICS
# ============================================================

metrics_df = pd.DataFrame(all_metrics)

metrics_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "metrics",
        "all_metrics.csv"
    ),

    index=False
)

# ============================================================
# SUMMARY
# ============================================================

summary = (

    metrics_df

    .groupby(["feature_set", "model"])[

        [
            "roc_auc",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1"
        ]

    ]

    .agg(["mean", "std"])
)

summary.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "metrics",
        "summary_metrics.csv"
    )
)

# ============================================================
# DONE
# ============================================================

print("\n" + "=" * 80)
print("PIPELINE COMPLETE")
print("=" * 80)

### Out-of-domain experiment

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd

import shap

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import StratifiedGroupKFold

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

MASTER_DATASET = # final master dataset containing all feature sets and sv groups
OUTPUT_DIR =

RANDOM_STATE = 42
N_SPLITS = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# LOAD DATA
# ============================================================

master_df = pd.read_parquet(MASTER_DATASET)
master_df["label"] = (master_df["flag"] == "ai").astype(int)

# ============================================================
# FEATURE SETS
# ============================================================

librosa_cols = [c for c in master_df.columns if c.startswith("librosa_")]
rqa_cols = [c for c in master_df.columns if c.startswith("rqa_")]
clap_cols = [c for c in master_df.columns if c.startswith("clap_")]

# ============================================================
# CLEAN RQA FEATURES
# ============================================================

master_df[rqa_cols] = master_df[rqa_cols].replace([np.inf, -np.inf], np.nan)
master_df[rqa_cols] = master_df[rqa_cols].fillna(master_df[rqa_cols].median())

prefixes = ('rqa_RR_')
cols_to_drop = [col for col in master_df.columns if col.startswith(prefixes)]
master_df.drop(columns=cols_to_drop, inplace=True)

rqa_cols = [c for c in master_df.columns if c.startswith("rqa_")]

FEATURE_SETS = {
    "librosa": librosa_cols,
    "rqa": rqa_cols,
    "clap": clap_cols,
    'librosa_rqa': librosa_cols + rqa_cols,
    'all': librosa_cols + clap_cols + rqa_cols,
}

# ============================================================
# MODEL (LOGISTIC REGRESSION PIPELINE)
# ============================================================

def build_model():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf",
            LogisticRegression(
                C=1.0,
                max_iter=3000,
                penalty="l2",
                solver="lbfgs",
                random_state=RANDOM_STATE
            )
        )
    ])

# ============================================================
# METRICS
# ============================================================

def compute_metrics(y_true, y_pred, y_prob):
    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0)
    }

# ============================================================
# SPLIT FUNCTION
# ============================================================

def stratified_group_holdout(df):
    sgkf = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    train_idx, test_idx = next(
        sgkf.split(df, df["label"], groups=df["cv_group"])
    )

    return df.iloc[train_idx], df.iloc[test_idx]

# ============================================================
# SHAP FUNCTION
# ============================================================

def compute_shap_values(model, X_test, feature_cols, metadata):

    scaler = model.named_steps["scaler"]
    clf = model.named_steps["clf"]

    X_test_scaled = scaler.transform(X_test)

    explainer = shap.LinearExplainer(clf, X_test_scaled)
    shap_values = explainer.shap_values(X_test_scaled)

    shap_df = pd.DataFrame(shap_values, columns=feature_cols)

    shap_long = shap_df.copy()
    shap_long["sample_index"] = X_test.index
    shap_long = shap_long.melt(
        id_vars="sample_index",
        var_name="feature",
        value_name="shap_value"
    )

    for k, v in metadata.items():
        shap_long[k] = v

    return shap_long

# ============================================================
# STORAGE
# ============================================================

all_results = []
shap_outputs = []
all_predictions = []


# ============================================================
# 1. LEAVE-ONE-SOURCE-OUT
# ============================================================

print("\n" + "=" * 80)
print("LEAVE-ONE-SOURCE-OUT")
print("=" * 80)

sources = master_df["source"].dropna().unique()

for held_out_source in sources:

    print("\nHELD OUT:", held_out_source)

    held_out_flag = master_df[
        master_df["source"] == held_out_source
    ]["flag"].iloc[0]

    if held_out_flag == "human":

        human_train = master_df[
            (master_df["flag"] == "human") &
            (master_df["source"] != held_out_source)
        ]

        human_test = master_df[
            master_df["source"] == held_out_source
        ]

        ai_df = master_df[
            master_df["flag"] == "ai"
        ]

        ai_train, ai_test = stratified_group_holdout(
            ai_df
        )

    else:

        ai_train = master_df[
            (master_df["flag"] == "ai") &
            (master_df["source"] != held_out_source)
        ]

        ai_test = master_df[
            master_df["source"] == held_out_source
        ]

        human_df = master_df[
            master_df["flag"] == "human"
        ]

        human_train, human_test = stratified_group_holdout(
            human_df
        )

    train_df = pd.concat([
        human_train,
        ai_train
    ])

    test_df = pd.concat([
        human_test,
        ai_test
    ])

    for feature_set_name, feature_cols in FEATURE_SETS.items():

        X_train = train_df[feature_cols]
        X_test = test_df[feature_cols]

        y_train = train_df["label"]
        y_test = test_df["label"]

        model = build_model()

        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(
            X_test
        )

        y_prob = model.predict_proba(
            X_test
        )[:,1]

        # ====================================================
        # SAVE PREDICTIONS
        # ====================================================

        pred_df = pd.DataFrame({

            "filename":
                test_df.index,

            "source":
                test_df["source"].values,

            "flag":
                test_df["flag"].values,

            "experiment":
                "leave_one_source_out",

            "held_out_source":
                held_out_source,

            "feature_set":
                feature_set_name,

            "y_true":
                y_test.values,

            "y_pred":
                y_pred,

            "y_prob":
                y_prob
        })

        all_predictions.append(
            pred_df
        )

        metrics = compute_metrics(
            y_test,
            y_pred,
            y_prob
        )

        all_results.append({

            "experiment":
                "leave_one_source_out",

            "held_out_source":
                held_out_source,

            "feature_set":
                feature_set_name,

            **metrics
        })

        # ===== SHAP =====

        shap_df = compute_shap_values(

            model,
            X_test,
            feature_cols,

            {
                "experiment":
                    "leave_one_source_out",

                "held_out_source":
                    held_out_source,

                "feature_set":
                    feature_set_name
            }
        )

        shap_outputs.append(
            shap_df
        )


# ============================================================
# SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(
    all_results
)

results_path = os.path.join(
    OUTPUT_DIR,
    "ood_results.csv"
)

results_df.to_csv(
    results_path,
    index=False
)


# ============================================================
# SAVE PREDICTIONS
# ============================================================

predictions_df = pd.concat(
    all_predictions,
    ignore_index=True
)

predictions_path = os.path.join(
    OUTPUT_DIR,
    "ood_predictions.csv"
)

predictions_df.to_csv(
    predictions_path,
    index=False
)

print(
    "Saved predictions:",
    predictions_path
)


# ============================================================
# SAVE SHAP VALUES
# ============================================================

shap_df_all = pd.concat(
    shap_outputs,
    ignore_index=True
)

shap_path = os.path.join(
    OUTPUT_DIR,
    "shap_values_long.csv"
)

shap_df_all.to_csv(
    shap_path,
    index=False
)


# ============================================================
# GLOBAL SHAP IMPORTANCE
# ============================================================

global_shap = (

    shap_df_all

    .groupby([
        "experiment",
        "feature_set",
        "feature"
    ])["shap_value"]

    .apply(
        lambda x:
        np.mean(np.abs(x))
    )

    .reset_index()

    .rename(
        columns={
            "shap_value":
            "mean_abs_shap"
        }
    )
)

global_path = os.path.join(
    OUTPUT_DIR,
    "shap_global_importance.csv"
)

global_shap.to_csv(
    global_path,
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

summary = (

    results_df

    .groupby([
        "experiment",
        "feature_set"
    ])[[

        "roc_auc",
        "accuracy",
        "balanced_accuracy",
        "f1"

    ]]

    .mean()
)

print("\nSUMMARY")
print(summary)

print("\nSaved results:", results_path)
print("Saved SHAP:", shap_path)
print("Saved SHAP global importance:", global_path)
print("Saved predictions:", predictions_path)